In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("movies.csv")

# Dataset shape
print("Dataset Shape:", df.shape)

# Column names
print("\nColumn Names:")
print(df.columns.tolist())

# First 5 rows
print("\nFirst 5 Rows:")
display(df.head())

# Dataset information
print("\nDataset Information:")
df.info()

# Identify text columns
print("\nText Columns:")
print(df.select_dtypes(include="object").columns.tolist())

Dataset Shape: (9742, 3)

Column Names:
['movieId', 'title', 'genres']

First 5 Rows:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB

Text Columns:
['title', 'genres']


/var/folders/56/vsxldn9s48v402t3nwb6fjrw0000gp/T/ipykernel_39419/1302849762.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include="object").columns.tolist())


In [3]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    
    return " ".join(words)

# Apply preprocessing on genres column
df["clean_text"] = df["genres"].apply(clean_text)

# Display original and cleaned text
display(df[["genres", "clean_text"]].head(10))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/tubanehal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,genres,clean_text
0,Adventure|Animation|Children|Comedy|Fantasy,adventure animation children comedy fantasy
1,Adventure|Children|Fantasy,adventure children fantasy
2,Comedy|Romance,comedy romance
3,Comedy|Drama|Romance,comedy drama romance
4,Comedy,comedy
5,Action|Crime|Thriller,action crime thriller
6,Comedy|Romance,comedy romance
7,Adventure|Children,adventure children
8,Action,action
9,Action|Adventure|Thriller,action adventure thriller


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Convert clean text into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df["clean_text"])

# Display TF-IDF matrix shape
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (9742, 177)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity between all movies
similarity_matrix = cosine_similarity(tfidf_matrix, dense_output=False)

# Display similarity matrix shape
print("Similarity Matrix Shape:", similarity_matrix.shape)

# Brief explanation
print("\nWhy Cosine Similarity is used:")
print("Cosine similarity measures how similar two movies are based on their text features.")
print("It compares the direction of their TF-IDF vectors, so movies with similar genres get higher similarity.")

Similarity Matrix Shape: (9742, 9742)

Why Cosine Similarity is used:
Cosine similarity measures how similar two movies are based on their text features.
It compares the direction of their TF-IDF vectors, so movies with similar genres get higher similarity.


In [6]:
def recommend(item_name, top_n=5):
    # Find the index of the selected movie
    matches = df[df["title"].str.lower() == item_name.lower()]

    if matches.empty:
        print("Movie not found.")
        return

    index = matches.index[0]

    # Get similarity scores
    scores = similarity_matrix[index].toarray().flatten()

    # Sort by similarity score
    similar_indices = scores.argsort()[::-1]

    # Remove the selected movie itself
    similar_indices = [i for i in similar_indices if i != index][:top_n]

    # Return top recommendations
    recommendations = df.iloc[similar_indices][["title", "genres"]].copy()
    recommendations["similarity_score"] = scores[similar_indices]

    return recommendations


# Test with 3 different movies
print("Recommendations for Toy Story (1995):")
display(recommend("Toy Story (1995)", 5))

print("Recommendations for Jumanji (1995):")
display(recommend("Jumanji (1995)", 5))

print("Recommendations for Grumpier Old Men (1995):")
display(recommend("Grumpier Old Men (1995)", 5))

Recommendations for Toy Story (1995):


,title,genres,similarity_score
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0


Recommendations for Jumanji (1995):


,title,genres,similarity_score
109,"NeverEnding Story III, The (1994)",Adventure|Children|Fantasy,1.0
8800,Pan (2015),Adventure|Children|Fantasy,1.0
9294,Alice Through the Looking Glass (2016),Adventure|Children|Fantasy,1.0
8641,Seventh Son (2014),Adventure|Children|Fantasy,1.0
7426,Alice in Wonderland (1933),Adventure|Children|Fantasy,1.0


Recommendations for Grumpier Old Men (1995):


,title,genres,similarity_score
1162,Addicted to Love (1997),Comedy|Romance,1.0
2031,American Pie (1999),Comedy|Romance,1.0
727,My Man Godfrey (1936),Comedy|Romance,1.0
361,Barcelona (1994),Comedy|Romance,1.0
4435,"Anarchist Cookbook, The (2002)",Comedy|Romance,1.0
